# Graph RAG eval v2 — COSORA (Fase 5)

Comparativa **template / llm / hybrid** sobre gold set + métricas de retrieval.

**Prerequisito:** `kg_ingest_v2` (Fase 1–3) + Neo4j cargado (`neo4j_graph_rag_v2` §3) o `EVAL_LOAD_NEO4J=True`.

### Colab + Drive

### Checklist antes del eval

1. **Neo4j** ya cargado (`neo4j_graph_rag_v2` §3) → `EVAL_LOAD_NEO4J=False` (no recargar en cada eval)
2. **Gold v2** — repo `docs/rag_eval_queries.json` o Drive `docs_queries/rag_eval_queries.json`
3. **`OPENAI_API_KEY`** en `.env` (rutas `llm` / `hybrid`)
4. **Chroma + graph/** en Drive (`catalog.json`, batches v2, `chroma_db`)
5. Opcional: `annotate_gold_chunks.py --write` → `source_chunk_id` en Q1–Q17

| Flag | Default |
|------|---------|
| `CYPHER_ROUTES_TO_TEST` | `["template", "llm", "hybrid"]` |
| `EVAL_LOAD_NEO4J` | `True` — carga Neo4j si abres solo este notebook |
| `WIPE_NEO4J_ON_EVAL` | `False` |
| `TOP_N` | `5` |


## 0. Setup + gold

In [ ]:
%pip install -q neo4j chromadb sentence-transformers rank_bm25 python-dotenv pandas openai

import json
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Flags eval ──────────────────────────────────────────────────────────
RUN_EVAL = True
CYPHER_ROUTES_TO_TEST = ["template", "llm", "hybrid"]
EVAL_LOAD_NEO4J = True              # True si ejecutas eval sin neo4j notebook previo
WIPE_NEO4J_ON_EVAL = False          # True solo si quieres wipe antes de eval
GRAPH_LOAD_MODE = "catalog_v2"
TOP_N = 5
RETRIEVAL_K = 50
RRF_K = 60
CYPHER_LIMIT = 50
CYPHER_LLM_MODEL = "gpt-4o-mini"
COLLECTION_NAME = "cosora_actas_e5"
SCHEMA_VERSION = 2
BRIDGE_MODE = "provenance"
MAX_GOLD = None                     # ej. 5 para prueba rápida

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None


def project_root() -> Path:
    nb_dir = Path(".").resolve()
    if nb_dir.name == "experiments":
        return nb_dir.parents[1]
    if nb_dir.name == "notebooks":
        return nb_dir.parent
    return nb_dir


def resolve_paths(runtime: str):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        root = project_root()
        docs_dir = str(root / "data" / "raw")
        chroma_path = str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    return docs_dir, chroma_path, graph_dir


DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)
CATALOG_PATH = Path(GRAPH_DIR) / "catalog.json"
EVAL_REPORT_PATH = Path(GRAPH_DIR) / "eval_report_v2.json"

def resolve_gold_path(runtime: str) -> Path:
    """Repo v2 gana sobre Drive v1 (evita evaluar gold antiguo en Colab)."""
    root = project_root()
    repo = root / "docs" / "rag_eval_queries.json"
    drive = (
        Path("/content/drive/MyDrive/RAG_UPC_Final_project/docs_queries/rag_eval_queries.json")
        if runtime == "colab"
        else None
    )
    candidates = [p for p in (repo, drive) if p is not None and p.exists()]
    if not candidates:
        return repo

    def _schema_ver(path: Path) -> int:
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            if isinstance(data, dict):
                return int(data.get("schema_version", 1))
        except Exception:
            pass
        return 1

    best = max(candidates, key=_schema_ver)
    if drive and drive.exists() and best != drive and _schema_ver(drive) < _schema_ver(repo):
        print(f"⚠️  Drive gold v{_schema_ver(drive)} ignorado — usando repo v{_schema_ver(repo)}: {repo}")
    return best


GOLD_PATH = resolve_gold_path(RUNTIME)

def load_gold_queries(path: Path) -> tuple[dict | None, list[dict]]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "queries" in data:
        return data, data["queries"]
    if isinstance(data, list):
        return None, data
    raise ValueError(f"Formato gold no reconocido: {path}")


_gold_meta, gold = load_gold_queries(GOLD_PATH)
if MAX_GOLD:
    gold = gold[:MAX_GOLD]

GOLD_SCHEMA_V2 = isinstance(_gold_meta, dict) and _gold_meta.get("schema_version", 0) >= 2

print(f"RUNTIME={RUNTIME}")
print(f"Gold: {len(gold)} queries ← {GOLD_PATH}  (schema={'v2' if GOLD_SCHEMA_V2 else 'v1'})")
print(f"GRAPH_DIR={GRAPH_DIR}")
print(f"Rutas Cypher: {CYPHER_ROUTES_TO_TEST}")
print(f"EVAL_LOAD_NEO4J={EVAL_LOAD_NEO4J}  WIPE={WIPE_NEO4J_ON_EVAL}")


## 1. Stack Neo4j + Chroma + retrieval

In [ ]:
import math
import re
import unicodedata

import chromadb
import pandas as pd
from neo4j import GraphDatabase
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

LOAD_TO_NEO4J = EVAL_LOAD_NEO4J
WIPE_NEO4J = WIPE_NEO4J_ON_EVAL
DOC_PREFIX = "passage: "
QUERY_PREFIX = "query: "
BM25_PATH = Path(CHROMA_PATH) / "bm25.json"

SCHEMA_CONSTRAINTS = [
    "CREATE CONSTRAINT entity_norm IF NOT EXISTS FOR (e:Entity) REQUIRE e.norm IS UNIQUE",
]
LOAD_CYPHER_V2 = """
UNWIND $rows AS row
MERGE (s:Entity {norm: row.s_norm})
  ON CREATE SET s.name = row.subject, s.batch = row.batch
  ON MATCH SET s.name = coalesce(s.name, row.subject)
MERGE (o:Entity {norm: row.o_norm})
  ON CREATE SET o.name = row.object, o.batch = row.batch
  ON MATCH SET o.name = coalesce(o.name, row.object)
MERGE (s)-[r:RELATED {triple_idx: row.triple_idx, batch: row.batch}]->(o)
  SET r.predicate = row.predicate,
      r.source_doc = row.source_doc,
      r.source_chunk_id = row.source_chunk_id
"""

CYPHER_TRANSVERSAL = """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE ANY(kw IN $keywords WHERE r.predicate CONTAINS kw
       OR a.norm CONTAINS kw OR b.norm CONTAINS kw)
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT $limit
"""
CYPHER_AGG = """
MATCH ()-[r:RELATED]->()
RETURN r.predicate AS predicate, count(*) AS cnt
ORDER BY cnt DESC
LIMIT $limit
"""
ENTITY_KEYWORDS = [
    "talud", "ute", "constructora", "luminaria", "andén", "anden", "ar-29",
    "hormigonado", "zapata", "incidencia", "df", "deo", "taludes",
]
FORBIDDEN = re.compile(
    r"\b(CREATE|MERGE|SET|DELETE|DETACH|DROP|REMOVE|FOREACH|LOAD\s+CSV)\b", re.I
)
CYPHER_SCHEMA_PROMPT = """
SCHEMA Neo4j COSORA v2:
- (:Entity {name: string, norm: string, batch: string})
- (a:Entity)-[r:RELATED {
    predicate: string, batch: string,
    source_doc: string, source_chunk_id: string
  }]->(b:Entity)

REGLAS:
- Solo MATCH, OPTIONAL MATCH, WHERE, RETURN, ORDER BY, LIMIT
- Prohibido: CREATE, MERGE, SET, DELETE, DETACH, DROP, CALL db.*
- LIMIT <= 50
- Buscar entidades: a.norm CONTAINS 'kw' OR b.norm CONTAINS 'kw' (minúsculas)
- Une condiciones con OR; no uses AND entre entidad y predicado
- NO uses r.predicate CONTAINS con palabras de la pregunta (predicados genéricos: estado, ejecuta, tiene...)
- La info relevante está en subject/object (entidades), no en el predicado
- RETURN siempre: subject, predicate, object, source_doc, source_chunk_id

EJEMPLO BUENO:
P: ¿Qué incidencias hay sobre el talud?
Keywords: talud, incidencia
C:
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE a.norm CONTAINS 'talud' OR b.norm CONTAINS 'talud'
   OR a.norm CONTAINS 'incidencia' OR b.norm CONTAINS 'incidencia'
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object,
       r.source_doc AS source_doc, r.source_chunk_id AS source_chunk_id
LIMIT 50

EJEMPLO MAL (no hacer):
WHERE a.norm CONTAINS 'talud' AND r.predicate CONTAINS 'incidencia'
"""


def norm_entity(name: str) -> str:
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", name.lower().strip())


class BM25Log1(BM25Okapi):
    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)
        self.chunk_ids = list(chunk_ids)
        df = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (self.corpus_size - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text: str) -> list[str]:
        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())

    @classmethod
    def load(cls, path: Path) -> "BM25Log1":
        data = json.loads(path.read_text(encoding="utf-8"))
        bm25 = cls.__new__(cls)
        bm25.chunk_ids = data["chunk_ids"]
        bm25.idf = {k: float(v) for k, v in data["idf"].items()}
        bm25.doc_freqs = data["doc_freqs"]
        bm25.doc_len = data["doc_len"]
        bm25.avgdl = data["avgdl"]
        bm25.corpus_size = data["corpus_size"]
        bm25.k1 = data["k1"]
        bm25.b = data["b"]
        return bm25


def extract_seeds(query: str) -> list[str]:
    lowered = query.lower()
    seeds = [kw for kw in ENTITY_KEYWORDS if kw in lowered]
    return seeds or [t for t in re.findall(r"\w+", lowered) if len(t) > 3][:3]


def cypher_template_route(query: str):
    lowered = query.lower()
    if any(p in lowered for p in ("frecuent", "más común", "más frecuente", "cuáles son las")):
        return CYPHER_AGG, {"limit": 20}
    return CYPHER_TRANSVERSAL, {"keywords": extract_seeds(query) or ["incidencia"], "limit": CYPHER_LIMIT}


def validate_cypher(cypher: str) -> tuple[bool, str]:
    text = cypher.strip().rstrip(";")
    if not text.upper().startswith("MATCH"):
        return False, "no MATCH"
    if "RETURN" not in text.upper():
        return False, "no RETURN"
    if FORBIDDEN.search(text):
        return False, "forbidden op"
    if "LIMIT" not in text.upper():
        text += "\nLIMIT 50"
    return True, text


def generate_cypher_llm(query: str) -> str:
    if client is None:
        raise RuntimeError("OpenAI no disponible")
    keywords = extract_seeds(query)
    kw_hint = ", ".join(keywords) if keywords else "(extrae palabras clave de la pregunta)"
    prompt = (
        CYPHER_SCHEMA_PROMPT
        + f"\nKeywords detectadas: {kw_hint}\n"
        + "Usa SOLO estas keywords en a.norm CONTAINS o b.norm CONTAINS (minúsculas, unidas con OR).\n"
        + f"\nP: {query}\nC:\n"
    )
    resp = client.chat.completions.create(
        model=CYPHER_LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=400,
    )
    raw = (resp.choices[0].message.content or "").strip()
    raw = re.sub(r"^```(?:cypher)?\s*", "", raw, flags=re.I)
    return re.sub(r"\s*```$", "", raw).strip()


def resolve_cypher_query(query: str, route: str) -> tuple[str, dict, str]:
    route = route.lower()
    if route == "template":
        cy, pa = cypher_template_route(query)
        return cy, pa, "template"
    if route == "llm":
        raw = generate_cypher_llm(query)
        ok, fixed = validate_cypher(raw)
        if not ok:
            raise ValueError(fixed)
        return fixed, {}, "llm"
    if route == "hybrid":
        try:
            raw = generate_cypher_llm(query)
            ok, fixed = validate_cypher(raw)
            if ok:
                return fixed, {}, "llm"
        except Exception:
            pass
        cy, pa = cypher_template_route(query)
        return cy, pa, "template"
    raise ValueError(route)


def run_neo4j_query(drv, cypher, params=None):
    with drv.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(cypher, **(params or {}))]


def build_load_rows_v2(relations, batch):
    rows = []
    for idx, rel in enumerate(relations):
        if isinstance(rel, (list, tuple)):
            s, p, o = rel[0], rel[1], rel[2]
            sd, sc = "", ""
        else:
            s, p, o = rel["subject"], rel["predicate"], rel["object"]
            sd = rel.get("source_doc") or ""
            sc = rel.get("source_chunk_id") or ""
        rows.append({
            "triple_idx": idx, "subject": s, "s_norm": norm_entity(s),
            "predicate": p, "object": o, "o_norm": norm_entity(o),
            "batch": batch, "source_doc": sd, "source_chunk_id": sc,
        })
    return rows


def load_catalog_v2(drv):
    if WIPE_NEO4J:
        with drv.session(database=NEO4J_DATABASE) as s:
            s.run("MATCH (n) DETACH DELETE n")
        print("✅ Neo4j wipe")
    catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
    for b in catalog.get("batches", []):
        path = Path(GRAPH_DIR) / b["json_file"]
        if not path.exists():
            print(f"⚠️  falta {path.name}")
            continue
        gd = json.loads(path.read_text(encoding="utf-8"))
        tag = b.get("tag") or f"batch{b['batch_id']:02d}"
        rows = build_load_rows_v2(gd.get("relations", []), tag)
        with drv.session(database=NEO4J_DATABASE) as s:
            for i in range(0, len(rows), 100):
                s.run(LOAD_CYPHER_V2, rows=rows[i : i + 100])
        print(f"  ✅ {tag}: {len(rows)} triples")
    return catalog


# Neo4j + Chroma
driver = None
if LOAD_TO_NEO4J:
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Neo4j no configurado en .env")
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    load_catalog_v2(driver)
    print("✅ Neo4j cargado para eval")
elif not (NEO4J_URI and NEO4J_PASSWORD):
    raise ValueError("Configura Neo4j o EVAL_LOAD_NEO4J=True")
else:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✅ Neo4j conectado (sin recarga)")

_chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection = _chroma.get_collection(COLLECTION_NAME)
_data = collection.get(include=["documents", "metadatas"])
all_docs, all_metas = list(_data["documents"]), list(_data["metadatas"])
chunk_by_id = {m["chunk_id"]: (d, m) for d, m in zip(all_docs, all_metas)}
cid_to_idx = {m["chunk_id"]: i for i, m in enumerate(all_metas)}
bm25_v2 = BM25Log1.load(BM25_PATH) if BM25_PATH.exists() else BM25Log1(
    [BM25Log1.extract_terms(d) for d in all_docs], [m["chunk_id"] for m in all_metas]
)
embedder = SentenceTransformer("intfloat/multilingual-e5-base")
print(f"✅ Chroma {len(all_docs)} chunks")


def execute_cypher(query, route):
    route_eff = route.lower()
    cy, pa, ru = resolve_cypher_query(query, route)
    rows = run_neo4j_query(driver, cy, pa if pa else None)
    if route_eff == "hybrid" and not rows and ru == "llm":
        print("  hybrid: LLM devolvió 0 filas → fallback template")
        cy, pa = cypher_template_route(query)
        rows = run_neo4j_query(driver, cy, pa if pa else None)
        ru = "template"
    return rows, cy, ru


def chunks_from_provenance(records, top_n=TOP_N):
    seen, hits = set(), []
    for row in records:
        if "cnt" in row and "subject" not in row:
            continue
        cid = row.get("source_chunk_id") or ""
        if not cid or cid in seen or cid not in chunk_by_id:
            continue
        text, meta = chunk_by_id[cid]
        seen.add(cid)
        hits.append({"text": text, "meta": meta, "score": 0.0, "from_graph": True})
        if len(hits) >= top_n:
            break
    return hits


def rrf_merge(a, b, top_n=TOP_N):
    scores = {}
    for rank, hit in enumerate(a):
        cid = hit["meta"]["chunk_id"]
        scores.setdefault(cid, {**hit, "score": 0.0})
        scores[cid]["score"] += 1.0 / (RRF_K + rank + 1)
    for rank, hit in enumerate(b):
        cid = hit["meta"]["chunk_id"]
        scores.setdefault(cid, {**hit, "score": 0.0})
        scores[cid]["score"] += 1.0 / (RRF_K + rank + 1)
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


def retrieve_baseline(query, top_n=TOP_N):
    qv = embedder.encode(QUERY_PREFIX + query).tolist()
    dr = collection.query(query_embeddings=[qv], n_results=RETRIEVAL_K)
    dense = [{"text": d, "meta": m} for d, m in zip(dr["documents"][0], dr["metadatas"][0])]
    terms = BM25Log1.extract_terms(query)
    sc = bm25_v2.get_scores(terms)
    ranked = sorted(range(len(sc)), key=lambda i: sc[i], reverse=True)[:RETRIEVAL_K]
    bm25h = []
    for idx in ranked:
        cid = bm25_v2.chunk_ids[idx]
        i = cid_to_idx[cid]
        bm25h.append({"text": all_docs[i], "meta": all_metas[i]})
    return rrf_merge(dense, bm25h, top_n=top_n)


def retrieve_cypher_rag_v2(query, *, top_n=TOP_N, cypher_route="hybrid"):
    records, cypher_used, route_used = execute_cypher(query, cypher_route)
    graph_hits = chunks_from_provenance(records, top_n=top_n)
    baseline = retrieve_baseline(query, top_n=top_n)
    fallback = len(graph_hits) < 1
    merged = baseline if fallback else rrf_merge(graph_hits, baseline, top_n=top_n)
    debug = {
        "cypher": cypher_used,
        "cypher_route_used": route_used,
        "n_triples": len(records),
        "n_graph_chunks": len(graph_hits),
        "fallback_baseline": fallback,
    }
    return merged, records, debug


print("✅ Stack eval listo")


## 2. Métricas

In [ ]:
LEGACY_SOURCE_DOC = {"ALL", "MULTI", "NONE"}


def infer_eval_mode(row: dict) -> str:
    mode = row.get("eval_mode")
    if mode:
        return mode
    sd = (row.get("source_doc") or "").upper()
    if sd == "NONE":
        return "negative"
    if sd in ("MULTI", "ALL"):
        return "any_doc"
    if row.get("source_chunk_id"):
        return "single_chunk"
    return "single_doc"


def norm_doc_id(name: str) -> str:
    """Gold .docx → stem Chroma doc_id."""
    if not name:
        return ""
    return Path(name).stem.lower()


def chunk_in_hits(hits: list[dict], target_chunk: str, *, top_n: int = TOP_N):
    if not target_chunk:
        return None
    cids = [h["meta"]["chunk_id"] for h in hits[:top_n]]
    return target_chunk in cids


def doc_in_hits(hits: list[dict], target_doc: str, *, graph_only: bool = False, top_n: int = TOP_N):
    if not target_doc or target_doc.upper() in LEGACY_SOURCE_DOC:
        return None
    target = norm_doc_id(target_doc)
    for h in hits[:top_n]:
        if graph_only and not h.get("from_graph"):
            continue
        if norm_doc_id(h["meta"]["doc_id"]) == target:
            return True
    return False


def any_doc_in_hits(
    hits: list[dict],
    source_docs: list[str] | None,
    *,
    graph_only: bool = False,
    top_n: int = TOP_N,
):
    docs = [d for d in (source_docs or []) if d]
    if not docs:
        return None
    return any(doc_in_hits(hits, doc, graph_only=graph_only, top_n=top_n) for doc in docs)


def compute_primary_metric(hits: list[dict], row: dict, *, top_n: int = TOP_N):
    mode = infer_eval_mode(row)
    target_chunk = row.get("source_chunk_id") or row.get("relevant_chunk_id") or ""

    if mode == "single_chunk":
        chunk_ok = chunk_in_hits(hits, target_chunk, top_n=top_n)
        doc_ok = doc_in_hits(hits, row.get("source_doc") or "", top_n=top_n)
        primary = chunk_ok if target_chunk else doc_ok
        return mode, primary, doc_ok, chunk_ok

    if mode == "single_doc":
        doc_ok = doc_in_hits(hits, row.get("source_doc") or "", top_n=top_n)
        chunk_ok = chunk_in_hits(hits, target_chunk, top_n=top_n) if target_chunk else None
        return mode, doc_ok, doc_ok, chunk_ok

    if mode in ("any_doc", "cross_doc_aggregate"):
        doc_ok = any_doc_in_hits(hits, row.get("source_docs"), top_n=top_n)
        return mode, doc_ok, doc_ok, None

    if mode in ("negative", "skip"):
        return mode, None, None, None

    doc_ok = doc_in_hits(hits, row.get("source_doc") or "", top_n=top_n)
    return mode, doc_ok, doc_ok, None


def bridge_doc_hit(hits: list[dict], row: dict, *, top_n: int = TOP_N):
    mode = infer_eval_mode(row)
    if mode == "single_doc" or mode == "single_chunk":
        return doc_in_hits(hits, row.get("source_doc") or "", graph_only=True, top_n=top_n)
    if mode in ("any_doc", "cross_doc_aggregate"):
        return any_doc_in_hits(hits, row.get("source_docs"), graph_only=True, top_n=top_n)
    if row.get("expects_graph_bridge"):
        docs = row.get("source_docs") or []
        if row.get("source_doc"):
            docs = docs + [row["source_doc"]]
        return any_doc_in_hits(hits, docs, graph_only=True, top_n=top_n)
    return None


def eval_one_query(row: dict, route: str) -> dict:
    qid = row["id"]
    query = row["query"]
    eval_mode = infer_eval_mode(row)

    out = {
        "id": qid,
        "route": route,
        "query": query,
        "eval_mode": eval_mode,
        "target_doc": row.get("source_doc") or "",
    }
    try:
        hits, records, dbg = retrieve_cypher_rag_v2(query, cypher_route=route, top_n=TOP_N)
        mode, primary_ok, doc_recall, chunk_recall = compute_primary_metric(hits, row)
        out.update(
            cypher_ok=True,
            cypher_route_used=dbg.get("cypher_route_used"),
            n_triples=dbg.get("n_triples", 0),
            n_graph_chunks=dbg.get("n_graph_chunks", 0),
            fallback_baseline=dbg.get("fallback_baseline"),
            cypher_nonempty=dbg.get("n_triples", 0) > 0,
            primary_ok=primary_ok,
            doc_recall_at_k=doc_recall,
            chunk_recall_at_k=chunk_recall,
            bridge_doc_hit=bridge_doc_hit(hits, row),
            expects_graph_bridge=bool(row.get("expects_graph_bridge")),
        )
        out["top_chunk_ids"] = [h["meta"]["chunk_id"] for h in hits[:TOP_N]]
    except Exception as exc:
        out.update(
            cypher_ok=False,
            error=str(exc),
            primary_ok=False,
            doc_recall_at_k=False,
            bridge_doc_hit=False,
        )
    return out


print("✅ Métricas eval v2 (por eval_mode)")


## 3. Ejecutar eval (`RUN_EVAL=True`)

In [ ]:
rows = []
eval_df = None

if not RUN_EVAL:
    print("RUN_EVAL=False — salto")
else:
    from IPython.display import display

    for route in CYPHER_ROUTES_TO_TEST:
        print(f"\\n=== Ruta: {route} ===")
        for i, g in enumerate(gold, 1):
            r = eval_one_query(g, route)
            rows.append(r)
            if r.get("eval_mode") in ("negative", "skip"):
                mark = "-"
            elif r.get("primary_ok") is True:
                mark = "✓"
            elif r.get("primary_ok") is False:
                mark = "✗"
            else:
                mark = "?"
            print(
                f"  [{mark}] {r['id']} ({r.get('eval_mode')}) "
                f"triples={r.get('n_triples', '?')} "
                f"graph_ch={r.get('n_graph_chunks', '?')} fb={r.get('fallback_baseline', '?')}"
            )

    eval_df = pd.DataFrame(rows)
    scored = eval_df[eval_df["primary_ok"].notna()]
    print("\\n--- Por ruta (primary_ok, excl. negative/skip) ---")
    display(scored.groupby("route").agg(
        n=("id", "count"),
        primary_ok_pct=("primary_ok", lambda s: round(100 * s.mean(), 1)),
        doc_recall_pct=("doc_recall_at_k", lambda s: round(100 * s.fillna(False).mean(), 1)),
        bridge_hit_pct=("bridge_doc_hit", lambda s: round(100 * s.fillna(False).mean(), 1)),
        fallback_pct=("fallback_baseline", lambda s: round(100 * s.fillna(False).mean(), 1)),
        avg_triples=("n_triples", "mean"),
    ))
    print("\\n--- Por eval_mode × ruta ---")
    display(scored.groupby(["eval_mode", "route"]).agg(
        n=("id", "count"),
        primary_ok_pct=("primary_ok", lambda s: round(100 * s.mean(), 1)),
    ))


## 4. Reporte → `eval_report_v2.json`

In [ ]:
if RUN_EVAL and len(rows):
    scored = eval_df[eval_df["primary_ok"].notna()]
    summary = {}
    by_mode = {}
    for route in CYPHER_ROUTES_TO_TEST:
        sub = eval_df[eval_df["route"] == route]
        sub_scored = scored[scored["route"] == route]
        summary[route] = {
            "n_queries": int(len(sub)),
            "n_scored": int(len(sub_scored)),
            "cypher_ok_pct": round(100 * sub["cypher_ok"].mean(), 1),
            "cypher_nonempty_pct": round(100 * sub["cypher_nonempty"].fillna(False).mean(), 1),
            "primary_ok_pct": round(100 * sub_scored["primary_ok"].mean(), 1) if len(sub_scored) else None,
            "doc_recall_at_k_pct": round(100 * sub_scored["doc_recall_at_k"].fillna(False).mean(), 1) if len(sub_scored) else None,
            "bridge_doc_hit_pct": round(100 * sub["bridge_doc_hit"].fillna(False).mean(), 1),
            "fallback_pct": round(100 * sub["fallback_baseline"].fillna(False).mean(), 1),
            "avg_n_triples": float(sub["n_triples"].mean()),
            "avg_n_graph_chunks": float(sub["n_graph_chunks"].mean()),
        }
        for mode, grp in sub_scored.groupby("eval_mode"):
            by_mode.setdefault(mode, {})[route] = {
                "n": int(len(grp)),
                "primary_ok_pct": round(100 * grp["primary_ok"].mean(), 1),
            }
    report = {
        "schema_version": SCHEMA_VERSION,
        "gold_schema_version": 2 if GOLD_SCHEMA_V2 else 1,
        "gold_path": str(GOLD_PATH),
        "n_gold": len(gold),
        "top_n": TOP_N,
        "routes": CYPHER_ROUTES_TO_TEST,
        "summary_by_route": summary,
        "summary_by_eval_mode": by_mode,
        "details": rows,
    }
    EVAL_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"📄 eval_report_v2.json → {EVAL_REPORT_PATH}")
    best = max(
        ((r, s) for r, s in summary.items() if s.get("primary_ok_pct") is not None),
        key=lambda x: x[1]["primary_ok_pct"],
    )
    print(f"🏆 Mejor primary_ok@K: {best[0]} ({best[1]['primary_ok_pct']}%)")
else:
    print("Ejecuta §3 (RUN_EVAL=True) primero")


## 5. Siguiente (Fase 6)

1. Elegir ruta Cypher prod según `doc_recall_at_k_pct` en eval_report
2. Promover stack a `src/rag/graph/`
3. Opcional: LLM-judge secundario sobre `expected_answer` del gold
